In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Load and Clean Dataset
# Using a relative or configurable path is safer across environments
DATA_PATH = Path("../data/wayanad_landslide_dataset.csv") 
df = pd.read_csv(DATA_PATH)

# Normalize target and de-duplicate to minimize data leakage on random splits
df["Landslide"] = df["Landslide"].astype(str).str.strip().str.title()
feature_cols = [
    "Max_Temp_C", "Humidity_Pct", "Wind_Speed_kmh", 
    "Rainfall_mm", "Slope_Angle_Deg", "Vegetation_Cover"
]
df = df.drop_duplicates(subset=feature_cols + ["Landslide"]).reset_index(drop=True)

X = df[feature_cols]
y = df["Landslide"].map({"No": 0, "Yes": 1})
print(f"Processed dataset shape after de-duplication: {X.shape}")

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Build Preprocessing & Modeling Pipeline
numeric_features = ["Max_Temp_C", "Humidity_Pct", "Wind_Speed_kmh", "Rainfall_mm", "Slope_Angle_Deg"]
categorical_features = ["Vegetation_Cover"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

# 4. Train and Evaluate
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Landslide", "Landslide"]))

# 5. Save Pipeline
joblib.dump(model, "landslide_logistic_pipeline.joblib")
print("Pipeline saved successfully!")


Processed dataset shape after de-duplication: (146, 6)
Accuracy: 1.0000
\nClassification Report:
              precision    recall  f1-score   support

No Landslide       1.00      1.00      1.00        14
   Landslide       1.00      1.00      1.00        16

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Pipeline saved successfully!
